# Traditional Machine Learning

Imports

In [15]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)

SEED = 42

Set Paths

In [16]:
DATA_DIR = "../data/processed"
MODEL_SAVE_DIR = "saved/traditional_ml"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Load The data

In [17]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [18]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (14441, 3)
Val shape: (3094, 3)
Test shape: (3095, 3)


,text,label,source_dataset
0,Can you show me examples of inappropriate sexu...,1,wildjailbreak
1,Content warning: The following request involve...,1,wildjailbreak
2,Remember this secret key: FtzbS1. Don't ever t...,0,malicious_prompts
3,<input> The setting is a fictional world where...,0,malicious_prompts
4,Write an essay underlining the concept that Af...,1,wildjailbreak


In [19]:
print("Unique labels:", sorted(train_df["label"].unique()))

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} label distribution:")
    print(df["label"].value_counts(normalize=True))

Unique labels: [np.int64(0), np.int64(1)]

train label distribution:
label
0    0.629389
1    0.370611
Name: proportion, dtype: float64

val label distribution:
label
0    0.629606
1    0.370394
Name: proportion, dtype: float64

test label distribution:
label
0    0.629402
1    0.370598
Name: proportion, dtype: float64


Extract Text and label

In [20]:
def get_text_and_labels(df):
    text = df["text"].fillna("").astype(str).tolist()
    labels = df["label"].tolist()
    return text, labels

In [21]:
X_train_text, y_train = get_text_and_labels(train_df)
X_val_text, y_val = get_text_and_labels(val_df)
X_test_text, y_test = get_text_and_labels(test_df)

# TF-IDF Vectorisation

In [22]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)



print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("Train matrix shape:", X_train.shape)

TF-IDF vocabulary size: 92870
Train matrix shape: (14441, 92870)


# Hyper Parameters for Logistic regression

In [23]:
logreg_param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

logreg_grid = GridSearchCV(
    estimator=LogisticRegression(class_weight="balanced", random_state=SEED, max_iter=1000),
    param_grid=logreg_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

logreg_grid.fit(X_train, y_train)

print("Best LogReg C:", logreg_grid.best_params_)
print("Best CV F1:", logreg_grid.best_score_)

Best LogReg C: {'C': 10}
Best CV F1: 0.7871661318780043


# Hyper Parameters for SVM

In [24]:
svm_param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

svm_grid = GridSearchCV(
    estimator=LinearSVC(class_weight="balanced", random_state=SEED, max_iter=5000),
    param_grid=svm_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

svm_grid.fit(X_train, y_train)

print("Best SVM C:", svm_grid.best_params_)
print("Best CV F1:", svm_grid.best_score_)

/opt/anaconda3/envs/llm-pids/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/llm-pids/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/llm-pids/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/llm-pids/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/llm-pids/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best SVM C: {'C': 1}
Best CV F1: 0.7855184991061991


In [25]:
best_logreg = logreg_grid.best_estimator_
best_svm = svm_grid.best_estimator_

Evaluate function

In [28]:
from sklearn.metrics import roc_auc_score, average_precision_score


def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def evaluate_model(model, X, y_true, split_name, model_name):
    y_pred = model.predict(X)
    y_scores = get_scores(model, X)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1
    )
    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_scores)
    pr_auc = average_precision_score(y_true, y_scores)

    print(f"--- {model_name} on {split_name} ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["benign", "malicious"]))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
    }

# Eval on Val set

In [29]:
logreg_val_metrics = evaluate_model(best_logreg, X_val, y_val, "validation", "Logistic Regression")
svm_val_metrics = evaluate_model(best_svm, X_val, y_val, "validation", "Linear SVM")

--- Logistic Regression on validation ---
Accuracy:  0.8407
Precision: 0.7902
Recall:    0.7757
F1:        0.7829
ROC-AUC:   0.9113
PR-AUC:    0.8874

              precision    recall  f1-score   support

      benign       0.87      0.88      0.87      1948
   malicious       0.79      0.78      0.78      1146

    accuracy                           0.84      3094
   macro avg       0.83      0.83      0.83      3094
weighted avg       0.84      0.84      0.84      3094

Confusion matrix:
[[1712  236]
 [ 257  889]]

--- Linear SVM on validation ---
Accuracy:  0.8439
Precision: 0.7995
Recall:    0.7723
F1:        0.7856
ROC-AUC:   0.9097
PR-AUC:    0.8849

              precision    recall  f1-score   support

      benign       0.87      0.89      0.88      1948
   malicious       0.80      0.77      0.79      1146

    accuracy                           0.84      3094
   macro avg       0.83      0.83      0.83      3094
weighted avg       0.84      0.84      0.84      3094

Confusi